In [23]:
import mne
import numpy as np
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

# MNE log mesajlarını azalt
mne.set_log_level('WARNING')

In [24]:
RUNS = ['R03', 'R04', 'R07', 'R08', 'R11', 'R12']
SUBJECTS = range(1, 10)  # Denek numaraları 1'den 109'a kadar
DATA_DIR = '../physioNet_Dataset'  # Veri seti ana klasör yolu
L_FREQ = 0.5
H_FREQ = 40.0
T_MIN = 1.0
T_MAX = 4.0

In [25]:
# HAM VERİ YÜKLEME



for subject_id in SUBJECTS:

    print(f"\n[ADIM 1] Ham EEG Verisi Yükleme - S{subject_id:03d}")
    
    # Denek klasör adını oluştur (örn: S001, S002, ...)
    sub = f'S{subject_id:03d}'
    raws = []
    
    # Her bir koşu için EDF dosyasını yükle
    for run in RUNS:
        # Dosya yolu: örnek -> ../physioNet_Dataset/S001/S001R03.edf
        file_path = f'{DATA_DIR}/{sub}/{sub}{run}.edf'
        
        try:
            # EDF dosyasını oku
            raw = mne.io.read_raw_edf(file_path, preload=True, verbose=False)
            raws.append(raw)
            
        except FileNotFoundError:
            print(f"  ✗ {run} bulunamadı!")
            



[ADIM 1] Ham EEG Verisi Yükleme - S001

[ADIM 1] Ham EEG Verisi Yükleme - S002

[ADIM 1] Ham EEG Verisi Yükleme - S003

[ADIM 1] Ham EEG Verisi Yükleme - S004

[ADIM 1] Ham EEG Verisi Yükleme - S005

[ADIM 1] Ham EEG Verisi Yükleme - S006

[ADIM 1] Ham EEG Verisi Yükleme - S007

[ADIM 1] Ham EEG Verisi Yükleme - S008

[ADIM 1] Ham EEG Verisi Yükleme - S009


In [26]:
# VERİLERİ BİRLEŞTİR
if not raws:
        print("  ✗ Birleştirilecek veri yok!")
else:
    raws = mne.concatenate_raws(raws)


In [27]:
# Eski kanal isimlerini kaydet
old_names = raw.ch_names[:5]  # İlk 5 kanal

# Kanal isimlerini temizle: nokta kaldır ve büyük harfe çevir
# Örnek: 'Fc5.' -> 'FC5'
raw.rename_channels({ch: ch.rstrip('.').upper() for ch in raw.ch_names})

# Standart 10-20 elektrot pozisyon sistemini uygula
# Bu, her kanalın kafadaki konumunu tanımlar
raw.set_montage(
    mne.channels.make_standard_montage('standard_1020'),
    on_missing='ignore'  # Tanımlı olmayan kanalları atla
)
print(f"  ✓ Önceki isimler: {old_names}")
print(f"  ✓ Yeni isimler  : {raw.ch_names[:5]}")
print(f"  ✓ Montaj uygulandı: 'standard_1020'")

  ✓ Önceki isimler: ['Fc5.', 'Fc3.', 'Fc1.', 'Fcz.', 'Fc2.']
  ✓ Yeni isimler  : ['FC5', 'FC3', 'FC1', 'FCZ', 'FC2']
  ✓ Montaj uygulandı: 'standard_1020'


In [28]:
# BANT GEÇİREN FİLTRE

raw.filter(
        L_FREQ,           # Alt kesim frekansı
        H_FREQ,           # Üst kesim frekansı
        fir_design='firwin',  # FIR filtre tasarımı
        verbose=False     # Detaylı çıktı kapalı
    )

<RawEDF | S009R12.edf, 64 x 19680 (123.0 s), ~9.7 MiB, data loaded>

In [29]:
# Notch Filtresi (Çentik) 
raw.notch_filter(freqs=50.0)

<RawEDF | S009R12.edf, 64 x 19680 (123.0 s), ~9.7 MiB, data loaded>

In [30]:
# EVENT ÇIKARMA
events, event_dict = mne.events_from_annotations(raw, verbose=False)
for event_name, event_code in event_dict.items():
    count = np.sum(events[:, 2] == event_code)
    print(f"    - {event_name:4s} (kod {event_code}): {count} kez")
event_id = {k: v for k, v in event_dict.items() if k in ['T1', 'T2']}



    - T0   (kod 1): 15 kez
    - T1   (kod 2): 8 kez
    - T2   (kod 3): 7 kez


In [31]:
# EPOCH OLUŞTURMA

epochs = mne.Epochs(
        raw,
        events,           # Event zamanları
        event_id,         # Hangi event'leri kullanacağız
        tmin=T_MIN,        # Epoch başlangıcı
        tmax=T_MAX,        # Epoch sonu
        proj=True,        # SSP projektörlerini uygula
        picks='eeg',      # Sadece EEG kanalları
        baseline=None,    # Baseline düzeltme yok
        preload=True,     # Veriyi belleğe yükle
        verbose=False
    )

In [32]:
# KÖTÜ EPOCHLARI ÇIKARMA

epochs.drop_bad(verbose=False)


In [33]:
# NUMPY ARRAY OLUŞTURMA
X = epochs.get_data().astype(np.float32)


In [34]:
# ETİKET KODLAMA (LABEL ENCODING)

event_codes = epochs.events[:, -1]
    
# Label encoder oluştur ve fit et
le = LabelEncoder()
y = le.fit_transform(event_codes).astype(np.int64)

In [35]:
X

array([[[-3.83281113e-05, -2.85323695e-05,  1.09099474e-05, ...,
          2.12434429e-06, -2.64730388e-05,  1.21431742e-06],
        [-4.60650153e-05, -2.29860416e-05,  8.39108907e-06, ...,
          1.74779561e-05, -1.08199874e-05, -4.03535751e-06],
        [-2.44320472e-05, -7.77426885e-06,  1.47813962e-05, ...,
          3.24092480e-06, -1.54676873e-05, -1.02122603e-05],
        ...,
        [-3.30225812e-05, -1.15519097e-05, -9.39714937e-06, ...,
         -1.34023558e-05, -1.81369978e-05, -1.16350448e-05],
        [-2.06130426e-05, -3.98240263e-06, -4.58294426e-06, ...,
         -2.05881188e-05, -2.76379469e-05, -2.73395308e-05],
        [-2.28366698e-05, -4.52213271e-06, -2.41283715e-06, ...,
         -1.98104553e-05, -1.41728333e-05, -7.13664804e-06]],

       [[ 3.73178373e-05,  2.18810928e-05,  4.60843221e-05, ...,
         -5.20261638e-05, -2.94614420e-05, -6.22484185e-06],
        [ 1.43789775e-05,  4.48378660e-06,  2.84926409e-05, ...,
         -5.27161210e-05, -4.83104159e

In [37]:
y

array([1, 0, 0, 1, 0, 1, 1, 0, 0, 1, 0, 1, 0, 1, 0])